# Static Oracle Mapping

This notebook will overview what exactly is meant by a "static oracle mapping". This mapping allows us to determine the best client to send a datapoint to during active learning.

## What is an oracle?

Whenever we say "oracle", we really just mean a specific client that may be an expert for a certain class. With the mock dataset for this project, we consider that there are nine of these classes that are the multi-class targets for our model to predict. To simplify things for our project, we will only use 3 clients, meaning that each of the clients will be an "oracle" for three of the classes.

## What is the mapping?

A mapping is simply a way for us to relate a client to the specific classes it can serve as an oracle for. In this project, we define this ourselves when we introduce false data heterogenity (at dataset splitting time), but this can actually be obtained in a variety of ways. 

## Why is the mapping 'static'?

Well, as mentioned above, for our project we keep the mapping the same based on how we split the data. This means that we have a static mapping - it does not change at all once the data is split. However, in a real-world implementation, there are a variety of ways that this could be done:

- We could randomly select clients to be oracles for different classes each round
- We could ask each client to identify which categories it sees the most of based on its data (natural data heterogenity), and form a mapping on that
- We could test each client with sample data to see which it is best at predictnig
- We could create a new model that can learn (via machine learning) which clients are best for which categories

Plus many others! For this project, again, we just pre-define an oracle mapping statically during data splitting, but these are all cool ways to do it in the real world!

## Making our mapping

Our mapping is made in the `[Data Preparation Notebook](../data/HSSFLDON_DataPreparation.ipynb)! We can simply define a variable to store it for us!

In [1]:
clientMappings = {
    "Client_0": ["insult", "dehumanize", "violence"],
    "Client_1": ["genocide", "attack_defend", "respect"],
    "Client_2": ["sentiment", "humiliate", "hatespeech"]
}

## Using our mapping

To use this mapping, all we simply do is make a prediction for a datapoint and determine which oracle to use. For each possible oracle, we will calculate how many of the assigned classes are predicted for the datapoint. The oracle with the most predicted classes wins! If two or more oracles have the same number of predicted classes, then we will just default to the first oracle for simplicity.

### Mock datapoints
First we will make some mock datapoints. These are stored in a dictionary, where each entry is `dict[data_point_id] = datapoint`!

In [ ]:
datapoints = {
    1: "this text might be an insult",
    2: "this text might be respectful",
    3: "this text might be humiliating"
}

### Mock categories

Next we will make our mock categories. The order here matters as we will see later on!

In [3]:
categories = ["sentiment", "respect", "insult", "humiliate", "status", "dehumanize", "violence", "genocide", "attack_defend", "hatespeech"]

### Mock predictions

Next we will make some mock predictions for these. These are also simply stored in a dictionary, where each entry is `dict[data_point_id] = prediction`. The prediction is an array of 9 floats, which mocks the example output of a multi-hot classifier model. For each element in the array, the float at that position represents whether or not it is predicted as the associated category for that index!

In [11]:
predictions = {
    1: [0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
    2: [0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    3: [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
}

### Mock oracle assignment

This example is quite simple! Our model seems to have only predicted one oracle for each datapoint, so we will just assign each datapoint to that oracle!

In [12]:
for dp_identifier, dp_prediction in predictions.items():
    matched_oracles = []
    for category, is_present in zip(categories, dp_prediction):
        if is_present:
            for client, client_categories in clientMappings.items():
                if category in client_categories:
                    matched_oracles.append(client)

    print(f"Datapoint {dp_identifier} matches with oracles: {matched_oracles}")

Datapoint 1 matches with oracles: ['Client_0']
Datapoint 2 matches with oracles: ['Client_1']
Datapoint 3 matches with oracles: ['Client_2']


## What next?

Once the datapoints are assigned, they'll be shipped off to the oracle for active learning!